# Make your own virtual Icechunk store

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-icechunks/2026-ioos-hackathon/blob/main/notebooks/simple-virtual-icechunk.ipynb)

Five daily NetCDF files of the NOAA Climate Data Record AVHRR NDVI sit in a public
AWS bucket. We turn them into **one** Icechunk store on your own disk, without copying
any of the data: the store holds only metadata and pointers to byte ranges in the
original files.

Based on Rich Signell's NDVI notebook from the
[June 5 NMFS HackHours](https://nmfs-opensci.github.io/NMFSHackDays-2026/topics/2026-06-05/).

In [ ]:
%pip install -q "icechunk>=2.2" "virtualizarr>=2.8" "xarray>=2026.7" "zarr>=3.4" obstore obspec_utils h5py s3fs "dask[array]" matplotlib

## 1. Find the files

The bucket is public, so we list it anonymously.

In [ ]:
import warnings
import s3fs

warnings.filterwarnings("ignore", category=UserWarning)

fs = s3fs.S3FileSystem(anon=True)
urls = [f"s3://{p}" for p in sorted(fs.ls("noaa-cdr-ndvi-pds/data/2000"))[:5]]
urls

## 2. Tell VirtualiZarr how to reach them

An object store for the bucket, a registry that maps the `s3://` prefix to it, and a
parser for the file format. These files are NetCDF4, which is HDF5 inside.

In [ ]:
from obstore.store import from_url
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr.parsers import HDFParser

bucket = "s3://noaa-cdr-ndvi-pds/"
registry = ObjectStoreRegistry({bucket: from_url(bucket, region="us-east-1", skip_signature=True)})
parser = HDFParser()

## 3. Open one file virtually

Only the metadata is read. The small coordinates are loaded into memory; the big arrays,
like `NDVI`, stay as `ManifestArray`s: lists of where each chunk lives in the file.

In [ ]:
from virtualizarr import open_virtual_dataset

coords = ["time", "latitude", "longitude"]
vds = open_virtual_dataset(url=urls[0], parser=parser, registry=registry, loadable_variables=coords)
vds

## 4. Combine the five days

Open each file the same way and join them along `time`. We drop three small CRS and
bounds-dimension variables that get in the way of the join. The keyword arguments stop
xarray from comparing the virtual arrays, which it cannot do.

In [ ]:
import xarray as xr

vds_list = [
    open_virtual_dataset(url=u, parser=parser, registry=registry, loadable_variables=coords)
    .drop_vars(["nv", "ncrs", "crs"], errors="ignore")
    for u in urls
]
combined = xr.concat(vds_list, dim="time", data_vars="minimal", coords="minimal",
                     compat="override", combine_attrs="override")
combined

## 5. Write the store and commit

The store lives in a local folder. The *virtual chunk container* tells Icechunk that some
chunks live outside the store, in the NDVI bucket, and how to reach them.

In [ ]:
import shutil
import icechunk

path = "ndvi-icechunk"
shutil.rmtree(path, ignore_errors=True)   # start fresh each run

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=bucket,
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    )
)
repo = icechunk.Repository.create(icechunk.local_filesystem_storage(path), config)
repo.save_config()

session = repo.writable_session("main")
combined.vz.to_icechunk(session.store)
session.commit("Five days of NDVI CDR, January 2000")

## 6. Open it as a reader would

A reader opens the store and says, explicitly, that it may fetch chunks from the NDVI
bucket. Then it is just an xarray dataset.

In [ ]:
creds = icechunk.containers_credentials({bucket: icechunk.s3_credentials(anonymous=True)})
reader = icechunk.Repository.open(icechunk.local_filesystem_storage(path),
                                  authorize_virtual_chunk_access=creds)
ds = xr.open_zarr(reader.readonly_session("main").store, consolidated=False, chunks={})
ds

## 7. Read real data

Only the chunks covering the US Pacific Northwest are fetched from the original files.

In [ ]:
pnw = ds["NDVI"].sel(latitude=slice(50, 40), longitude=slice(-126, -114)).isel(time=0)
pnw.plot(cmap="YlGn", vmin=0, vmax=0.8);

## How big is the store?

In [ ]:
import os

size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs_ in os.walk(path) for f in fs_)
print(f"Icechunk store: {size / 1e3:.0f} kB")
print(f"NetCDF files it points at: {sum(fs.size(u) for u in urls) / 1e6:.0f} MB")